# DecoupleNet LoveDA Semantic Segmentation — Google Colab

Bu notebook DecoupleNet modelini LoveDA veri seti ile Google Colab'da egitir.

**Veri kaynagi:** HuggingFace (`chloechia/loveda`)

## Adimlar
1. Ortam kontrolu (GPU, disk)
2. Repo klonlama
3. Bagimlilik kurulumu
4. Pre-trained agirlik indir
5. LoveDA veri setini indir + hazirla
6. Veri setini dogrula
7. (Opsiyonel) Google Drive bagla
8. Egitimi baslat
9. TensorBoard ile izle
10. Sonuclari kaydet

In [ ]:
# ===================== HUCRE 1: ORTAM KONTROLU =====================
import torch
import subprocess

print("=" * 50)
print("ORTAM KONTROLU")
print("=" * 50)

# GPU kontrolu
gpu_available = torch.cuda.is_available()
print(f"GPU mevcut: {gpu_available}")
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("UYARI: GPU bulunamadi! Egitim cok yavas olacak.")
    print("Runtime -> Change runtime type -> T4 GPU secin")

# Disk alani
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(f"\nDisk alani:\n{result.stdout}")

# Python versiyonu
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===================== HUCRE 2: REPO KLONLAMA =====================
import os

if not os.path.exists('/content/DecoupleNet'):
    !git clone https://github.com/myrisee/DecoupleNet.git /content/DecoupleNet
    print("Repo klonlandi!")
else:
    print("Repo zaten mevcut.")

# segmentation dizinine git
%cd /content/DecoupleNet/segmentation
print(f"Calisma dizini: {os.getcwd()}")

In [ ]:
# ===================== HUCRE 3: BAGIMLILIK KURULUMU =====================
print("Bagimliliklar kuruluyor...")

# Ana bagimliliklar
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Proje bagimliliklari
!pip install -q timm catalyst==20.09 pytorch-lightning==1.9.0
!pip install -q albumentations einops ttach pytorch-toolbelt
!pip install -q opencv-python-headless scipy matplotlib tqdm addict
!pip install -q antialiased-cnns

# HuggingFace indirme icin
!pip install -q huggingface_hub

print("Bagimliliklar kuruldu!")

# Versiyon kontrolu
import torch, timm, pytorch_lightning as pl
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"Lightning: {pl.__version__}")

In [ ]:
# ===================== HUCRE 3b: COLAB DOSYALARINI OLUSTUR =====================
# Repo GitHub'dan klonlandigi icin bizim yeni dosyalarimiz yok.
# Bu hucre dataset loader + config + torch.load yamasi olusturur.

import os

SEG = '/content/DecoupleNet/segmentation'

# ---- 1. Flat dataset loader ----
loader_path = os.path.join(SEG, 'geoseg', 'datasets', 'loveda_dataset_colab.py')
with open(loader_path, 'w') as f:
    f.write('''"""
LoveDA Flat Dataset Loader for Google Colab.
Urban/Rural alt dizinleri gerektirmez; duz (flat) dizin yapisindan okur.
"""
from .transform import *
import os, os.path as osp, random
import numpy as np, torch, cv2
from torch.utils.data import Dataset
import albumentations as albu
from PIL import Image

CLASSES = ('background', 'building', 'road', 'water', 'barren', 'forest', 'agricultural')
PALETTE = [[255,255,255],[255,0,0],[255,255,0],[0,0,255],[159,129,183],[0,255,0],[255,195,128]]
ORIGIN_IMG_SIZE = (1024, 1024)

def get_training_transform():
    return albu.Compose([
        albu.HorizontalFlip(p=0.5), albu.VerticalFlip(p=0.5),
        albu.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.25),
        albu.Normalize()])

def train_aug(img, mask):
    crop_aug = Compose([RandomScale(scale_list=[0.75,1.0,1.25,1.5], mode='value'),
                        SmartCropV1(crop_size=512, max_ratio=0.75, ignore_index=255, nopad=False)])
    img, mask = crop_aug(img, mask)
    img, mask = np.array(img), np.array(mask)
    aug = get_training_transform()(image=img.copy(), mask=mask.copy())
    return aug['image'], aug['mask']

def get_val_transform():
    return albu.Compose([albu.Normalize()])

def val_aug(img, mask):
    img, mask = np.array(img), np.array(mask)
    aug = get_val_transform()(image=img.copy(), mask=mask.copy())
    return aug['image'], aug['mask']

class LoveDAFlatTrainDataset(Dataset):
    def __init__(self, data_root='/content/data/LoveDA/Train',
                 img_dir='images_png', mask_dir='masks_png_convert',
                 img_suffix='.png', mask_suffix='.png',
                 mosaic_ratio=0.25, transform=train_aug, img_size=ORIGIN_IMG_SIZE):
        self.data_root, self.img_dir, self.mask_dir = data_root, img_dir, mask_dir
        self.mosaic_ratio = mosaic_ratio
        self.img_suffix, self.mask_suffix = img_suffix, mask_suffix
        self.transform, self.img_size = transform, img_size
        self.img_ids = self._get_img_ids()

    def __getitem__(self, index):
        img, mask = self._load(index)
        if random.random() < self.mosaic_ratio:
            img, mask = self._mosaic(index)
        if self.transform:
            img, mask = self.transform(img, mask)
        img = torch.from_numpy(img).permute(2,0,1).float()
        mask = torch.from_numpy(mask).long()
        return {'img': img, 'gt_semantic_seg': mask, 'img_id': self.img_ids[index], 'img_type': 'all'}

    def __len__(self): return len(self.img_ids)

    def _get_img_ids(self):
        ip = osp.join(self.data_root, self.img_dir)
        mp = osp.join(self.data_root, self.mask_dir)
        assert osp.isdir(ip), f"Img dir not found: {ip}"
        assert osp.isdir(mp), f"Mask dir not found: {mp}"
        imgs = sorted(f.split('.')[0] for f in os.listdir(ip))
        masks = sorted(f.split('.')[0] for f in os.listdir(mp))
        assert len(imgs)==len(masks), f"Img({len(imgs)})!=Mask({len(masks)})"
        return imgs

    def _load(self, index):
        img_id = self.img_ids[index]
        img = Image.open(osp.join(self.data_root, self.img_dir, img_id+self.img_suffix)).convert('RGB')
        mask = Image.open(osp.join(self.data_root, self.mask_dir, img_id+self.mask_suffix)).convert('L')
        return img, mask

    def _mosaic(self, index):
        idxs = [index]+[random.randint(0,len(self.img_ids)-1) for _ in range(3)]
        imgs = [np.array(self._load(i)[0]) for i in idxs]
        masks = [np.array(self._load(i)[1]) for i in idxs]
        h, w = self.img_size
        ox, oy = random.randint(w//4,w-w//4), random.randint(h//4,h-h//4)
        cs = [(ox,oy),(w-ox,oy),(ox,h-oy),(w-ox,h-oy)]
        ic = [albu.RandomCrop(width=cw,height=ch)(image=a)['image'] for a,(cw,ch) in zip(imgs,cs)]
        top, bot = np.concatenate((ic[0],ic[1]),1), np.concatenate((ic[2],ic[3]),1)
        img = np.ascontiguousarray(np.concatenate((top,bot),0))
        mc = [albu.RandomCrop(width=cw,height=ch)(image=np.zeros((*a.shape,3),dtype=np.uint8),mask=a)['mask'] for a,(cw,ch) in zip(masks,cs)]
        topm, botm = np.concatenate((mc[0],mc[1]),1), np.concatenate((mc[2],mc[3]),1)
        mask = np.ascontiguousarray(np.concatenate((topm,botm),0))
        return Image.fromarray(img), Image.fromarray(mask)

loveda_val_dataset = LoveDAFlatTrainDataset(data_root='/content/data/LoveDA/Val', mosaic_ratio=0.0, transform=val_aug)
''')
print(f"[1/3] Olusturuldu: {loader_path}")

# ---- 2. Colab config ----
config_path = os.path.join(SEG, 'config', 'loveda', 'train_decouplenet_colab.py')
with open(config_path, 'w') as f:
    f.write('''"""LoveDA + DecoupleNet Config - Google Colab"""
from torch.utils.data import DataLoader
from geoseg.losses import UnetFormerLoss
from geoseg.datasets.loveda_dataset_colab import LoveDAFlatTrainDataset, CLASSES
from geoseg.models.UNetFormer_decouplenet import UNetFormer_DecoupleNet_D2
from catalyst.contrib.nn import Lookahead
from catalyst import utils
import torch, numpy as np, albumentations as albu
from geoseg.datasets.transform import Compose, RandomScale, SmartCropV1

max_epoch = 30
ignore_index = len(CLASSES)
train_batch_size = 4
val_batch_size = 4
lr = 4e-4
weight_decay = 0.01
backbone_lr = 4e-4
backbone_weight_decay = 0.01
num_classes = len(CLASSES)
classes = CLASSES
weights_name = "decouplenet-loveda-colab-epoch30"
weights_path = "/content/DecoupleNet/segmentation/model_weights/loveda"
test_weights_name = "last"
log_name = 'loveda/decouplenet-colab'
monitor = 'val_mIoU'
monitor_mode = 'max'
save_top_k = 1
save_last = True
check_val_every_n_epoch = 1
pretrained_ckpt_path = None
resume_ckpt_path = None
gpus = 1

net = UNetFormer_DecoupleNet_D2(num_classes=num_classes)
loss = UnetFormerLoss(ignore_index=ignore_index)
use_aux_loss = True

def get_training_transform():
    return albu.Compose([albu.HorizontalFlip(p=0.5), albu.VerticalFlip(p=0.5),
        albu.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.25), albu.Normalize()])

def train_aug(img, mask):
    crop_aug = Compose([RandomScale(scale_list=[0.75,1.0,1.25,1.5], mode='value'),
        SmartCropV1(crop_size=512, max_ratio=0.75, ignore_index=ignore_index, nopad=False)])
    img, mask = crop_aug(img, mask)
    img, mask = np.array(img), np.array(mask)
    aug = get_training_transform()(image=img.copy(), mask=mask.copy())
    return aug['image'], aug['mask']

def val_aug(img, mask):
    img, mask = np.array(img), np.array(mask)
    aug = albu.Compose([albu.Normalize()])(image=img.copy(), mask=mask.copy())
    return aug['image'], aug['mask']

train_dataset = LoveDAFlatTrainDataset(transform=train_aug, data_root='/content/data/LoveDA/Train', mosaic_ratio=0.25)
val_dataset = LoveDAFlatTrainDataset(transform=val_aug, data_root='/content/data/LoveDA/Val', mosaic_ratio=0.0)
train_loader = DataLoader(dataset=train_dataset, batch_size=train_batch_size, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=val_batch_size, num_workers=2, shuffle=False, pin_memory=True, drop_last=False)

layerwise_params = {"backbone.*": dict(lr=backbone_lr, weight_decay=backbone_weight_decay)}
net_params = utils.process_model_params(net, layerwise_params=layerwise_params)
base_optimizer = torch.optim.AdamW(net_params, lr=lr, weight_decay=weight_decay)
optimizer = Lookahead(base_optimizer)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epoch, eta_min=1e-6)
''')
print(f"[2/3] Olusturuldu: {config_path}")

# ---- 3. torch.load patch (PyTorch 2.6+ güvenlik) ----
decouple_path = os.path.join(SEG, 'geoseg', 'models', 'DecoupleNet.py')
with open(decouple_path, 'r') as f:
    content = f.read()

old = "torch.load('../backbone_weights/DecoupleNet_D2.pth',\n                            map_location=torch.device('cuda:0'))"
new = "torch.load('../backbone_weights/DecoupleNet_D2.pth',\n                            map_location=torch.device('cuda:0'), weights_only=False)"

if old in content:
    content = content.replace(old, new)
    with open(decouple_path, 'w') as f:
        f.write(content)
    print(f"[3/3] Yamalandi: {decouple_path} (weights_only=False eklendi)")
elif 'weights_only=False' in content:
    print(f"[3/3] Zaten yamali: {decouple_path}")
else:
    print(f"[3/3] UYARI: torch.load satiri bulunamadi, manuel kontrol gerek!")

print("\nTum Colab dosyalari hazir!")

In [ ]:
# ===================== HUCRE 4: PRE-TRAINED AGIRLIK INDIR =====================
import os

weights_dir = '/content/DecoupleNet/segmentation/backbone_weights'
weights_path = os.path.join(weights_dir, 'DecoupleNet_D2.pth')
weights_url = 'https://github.com/lwCVer/DecoupleNet/releases/download/weights/DecoupleNet_D2.pth'

os.makedirs(weights_dir, exist_ok=True)

if not os.path.exists(weights_path):
    print(f"Agirlik indiriliyor: {weights_url}")
    !wget -q --show-progress -O "{weights_path}" "{weights_url}"
    file_size = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Indirildi: {file_size:.1f} MB")
else:
    file_size = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Agirlik dosyasi zaten mevcut: {file_size:.1f} MB")

# PyTorch 2.6+ güvenlik: weights_only=False gerekiyor
import torch
try:
    ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
    print(f"Agirlik dogrulandi! Keys: {len(ckpt)} parametre")
    if 'model' in ckpt:
        print(f"Model parametre sayisi: {len(ckpt['model'])}")
except Exception as e:
    print(f"HATA: Agirlik dosyasi gecersiz: {e}")

In [ ]:
# ===================== HUCRE 5: LOVEDA VERI SETINI INDIR + HAZIRLA =====================
# Bu islem ~5-10 dakika surer (4.67 GB indirilecek)
# Script repo'da olmadigi icin burada olusturulur

%cd /content/DecoupleNet/segmentation

import os, glob, shutil, time, sys, numpy as np, cv2
from pathlib import Path

data_root = '/content/data/LoveDA'

# Eger veri zaten hazirsa atla
train_ready = os.path.exists(os.path.join(data_root, 'Train', 'masks_png_convert'))
val_ready = os.path.exists(os.path.join(data_root, 'Val', 'masks_png_convert'))

if train_ready and val_ready:
    print("Veri seti zaten hazir! Indirme atlaniyor.")
else:
    # HuggingFace hub kur
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        !pip install -q huggingface_hub
        from huggingface_hub import snapshot_download

    HF_REPO_ID = "chloechia/loveda"

    # Hedef dizinleri olustur
    train_img_dir = os.path.join(data_root, 'Train', 'images_png')
    train_mask_dir = os.path.join(data_root, 'Train', 'masks_png')
    val_img_dir = os.path.join(data_root, 'Val', 'images_png')
    val_mask_dir = os.path.join(data_root, 'Val', 'masks_png')
    for d in [train_img_dir, train_mask_dir, val_img_dir, val_mask_dir]:
        os.makedirs(d, exist_ok=True)

    # HuggingFace dizin isimleri
    download_configs = [
        ("urban:rural train images", train_img_dir, "Train Goruntuleri"),
        ("urban:rural train masks",  train_mask_dir, "Train Maskeleri"),
        ("urban:rural val images",   val_img_dir,    "Val Goruntuleri"),
        ("urban:rural val masks",    val_mask_dir,   "Val Maskeleri"),
    ]

    total_downloaded = 0
    for hf_dir, target_dir, description in download_configs:
        print(f"\n>>> {description} indiriliyor...")
        t0 = time.time()

        cache_path = snapshot_download(
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            allow_patterns=[f"{hf_dir}/*.png"],
        )

        src_dir = os.path.join(cache_path, hf_dir)
        if not os.path.isdir(src_dir):
            print(f"  HATA: {src_dir} bulunamadi!")
            print(f"  Mevcut dizinler: {os.listdir(cache_path)}")
            continue

        files = [f for f in os.listdir(src_dir) if f.endswith('.png')]
        for f in files:
            src = os.path.join(src_dir, f)
            dst = os.path.join(target_dir, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)

        total_downloaded += len(files)
        print(f"  {len(files)} dosya indirildi ({time.time()-t0:.1f}s)")

    print(f"\n>>> Toplam indirilen: {total_downloaded} dosya")

    # ===================== MASKE DONUSUMU =====================
    def convert_label(mask):
        mask[mask == 0] = 8
        mask -= 1
        return mask

    PALETTE = [[255,255,255],[255,0,0],[255,255,0],[0,0,255],
               [159,129,183],[0,255,0],[255,195,128]]

    def convert_masks(masks_dir, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        mask_paths = sorted(glob.glob(os.path.join(masks_dir, "*.png")))
        total = len(mask_paths)
        print(f"  Maske donusumu: {total} dosya isleniyor...")
        for i, mask_path in enumerate(mask_paths):
            mask_filename = os.path.splitext(os.path.basename(mask_path))[0]
            mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
            label = convert_label(mask)
            cv2.imwrite(os.path.join(output_dir, f"{mask_filename}.png"), label)
            if (i + 1) % 200 == 0 or (i + 1) == total:
                print(f"    [{i+1}/{total}] islendi")
        print(f"  Tamamlandi: {total} maske donusturuldu")
        return total

    print("\n>>> Maske donusumu baslatiliyor...")
    print("\n[1/2] Train maskeleri:")
    convert_masks(train_mask_dir, os.path.join(data_root, 'Train', 'masks_png_convert'))
    print("\n[2/2] Val maskeleri:")
    convert_masks(val_mask_dir, os.path.join(data_root, 'Val', 'masks_png_convert'))

    # ===================== DOGRULAMA =====================
    ti = len(glob.glob(os.path.join(train_img_dir, "*.png")))
    tm = len(glob.glob(os.path.join(data_root, 'Train', 'masks_png_convert', "*.png")))
    vi = len(glob.glob(os.path.join(val_img_dir, "*.png")))
    vm = len(glob.glob(os.path.join(data_root, 'Val', 'masks_png_convert', "*.png")))
    print(f"\n{'='*50}")
    print(f"Train: {ti} goruntu, {tm} maske {'OK' if ti==tm else 'HATA!'}")
    print(f"Val:   {vi} goruntu, {vm} maske {'OK' if vi==vm else 'HATA!'}")
    print(f"Toplam: {ti+vi} goruntu")

print("\nHazir!")

In [ ]:
# ===================== HUCRE 6: VERI SETINI DOGRULA =====================
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

data_root = '/content/data/LoveDA'

# Dosya sayilari
train_imgs = sorted(glob.glob(os.path.join(data_root, 'Train', 'images_png', '*.png')))
train_masks = sorted(glob.glob(os.path.join(data_root, 'Train', 'masks_png_convert', '*.png')))
val_imgs = sorted(glob.glob(os.path.join(data_root, 'Val', 'images_png', '*.png')))
val_masks = sorted(glob.glob(os.path.join(data_root, 'Val', 'masks_png_convert', '*.png')))

print("VERI SETI ISTATISTIKLERI")
print("=" * 40)
print(f"Train goruntuleri:  {len(train_imgs)}")
print(f"Train maskeleri:   {len(train_masks)}")
print(f"Val goruntuleri:    {len(val_imgs)}")
print(f"Val maskeleri:     {len(val_masks)}")
print(f"Toplam:            {len(train_imgs) + len(val_imgs)} goruntu")

# Ornek gorsellestirme
if len(train_imgs) > 0:
    img = Image.open(train_imgs[0]).convert('RGB')
    mask = Image.open(train_masks[0]).convert('L')
    mask_arr = np.array(mask)
    unique_vals = np.unique(mask_arr)
    print(f"\nOrnek goruntu boyutu: {img.size}")
    print(f"Maske deger araligi: {unique_vals}")

    CLASSES = ('background', 'building', 'road', 'water', 'barren', 'forest', 'agricultural')
    PALETTE = [[255,255,255], [255,0,0], [255,255,0], [0,0,255], [159,129,183], [0,255,0], [255,195,128]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img)
    axes[0].set_title(f'Goruntu: {os.path.basename(train_imgs[0])}')
    axes[0].axis('off')

    # Maskeyi renklendir
    mask_rgb = np.zeros((*mask_arr.shape, 3), dtype=np.uint8)
    for cls_idx, color in enumerate(PALETTE):
        if cls_idx in unique_vals:
            mask_rgb[mask_arr == cls_idx] = color
    axes[1].imshow(mask_rgb)
    axes[1].set_title(f'Maske: {os.path.basename(train_masks[0])}')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Sinif dagilimi
    print("\nSinif dagilimi (ilk ornek maske):")
    for val in unique_vals:
        if val < len(CLASSES):
            count = np.sum(mask_arr == val)
            pct = count / mask_arr.size * 100
            print(f"  {CLASSES[val]:15s}: {count:>8d} piksel ({pct:.1f}%)")

In [ ]:
# ===================== HUCRE 7: (OPSIYONEL) GOOGLE DRIVE BAGLA =====================
# Model checkpoint'lerini kalici olarak kaydetmek icin Google Drive kullanin

from google.colab import drive
import shutil

DRIVE_MOUNT = '/content/drive'
CHECKPOINT_DRIVE_DIR = '/content/drive/MyDrive/DecoupleNet_checkpoints'

# Google Drive bagla
if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)
    print("Google Drive baglandi!")
else:
    print("Google Drive zaten bagli.")

# Checkpoint dizinini olustur
os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)

# Symbolic link olustur: model_weights/loveda -> Drive
local_weights = '/content/DecoupleNet/segmentation/model_weights/loveda'
os.makedirs(os.path.dirname(local_weights), exist_ok=True)

if os.path.islink(local_weights):
    os.unlink(local_weights)
elif os.path.exists(local_weights):
    shutil.rmtree(local_weights)

os.symlink(CHECKPOINT_DRIVE_DIR, local_weights)
print(f"Checkpoint dizini: {local_weights} -> {CHECKPOINT_DRIVE_DIR}")
print("Egitim bitince checkpoint'ler Google Drive'da kalici olacak!")

In [ ]:
# ===================== HUCRE 8: EGITIMI BASLAT =====================
# Tahmini sure: T4 ile ~12-15 saat (30 epoch)
# Colab ucretsiz hesap 12 saat zaman asimi var
# Her epoch sonunda checkpoint kaydedilir

%cd /content/DecoupleNet/segmentation

!python train_supervision.py -c config/loveda/train_decouplenet_colab.py

In [ ]:
# ===================== HUCRE 9: TENSORBOARD ILE IZLE =====================
# Egitim sirasinda bu hucreyi calistirarak metrikleri izleyebilirsiniz

%load_ext tensorboard
%tensorboard --logdir /content/DecoupleNet/segmentation/lightning_logs

In [ ]:
# ===================== HUCRE 10: SONUCLARI KAYDET =====================
import os
import shutil
import glob

print("EGITIM SONUCLARI")
print("=" * 50)

# Checkpoint dosyalari
ckpt_dir = '/content/DecoupleNet/segmentation/model_weights/loveda'
if os.path.exists(ckpt_dir):
    ckpt_files = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    print(f"Checkpoint dosyalari ({len(ckpt_files)}):")
    for f in ckpt_files:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {os.path.basename(f):50s} {size_mb:.1f} MB")

# Google Drive'a kopyala (eger Drive bagliysa)
drive_ckpt_dir = '/content/drive/MyDrive/DecoupleNet_checkpoints'
if os.path.ismount('/content/drive') and os.path.exists(ckpt_dir):
    # Eger symbolic link degilse, manuel kopyala
    if not os.path.islink(ckpt_dir):
        os.makedirs(drive_ckpt_dir, exist_ok=True)
        for f in glob.glob(os.path.join(ckpt_dir, '*.ckpt')):
            dst = os.path.join(drive_ckpt_dir, os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy2(f, dst)
                print(f"  Kopyalandi: {os.path.basename(f)}")
        print("\nTum checkpoint'ler Google Drive'a kaydedildi!")
    else:
        print("\nCheckpoint'ler zaten Google Drive'a dogrudan kaydedildi (symlink).")

# Log dosyalari
log_dir = '/content/DecoupleNet/segmentation/lightning_logs'
if os.path.exists(log_dir):
    log_files = glob.glob(os.path.join(log_dir, '**', '*.csv'), recursive=True)
    print(f"\nLog dosyalari ({len(log_files)}):")
    for f in log_files:
        print(f"  {f}")

print("\nTamamlandi!")